# Lesson 17 Lab — Reading GPU Specification Tables Critically

**Puzzle:** Can one TFLOPS, TOPS, memory-capacity, or bandwidth number tell you which GPU is faster for an LLM workload?

This notebook retains one complete RTX 5090 execution.


## Why this matters

A specification only has meaning with its precision, dense/sparse convention, clock basis, form factor, memory technology, and workload connection. Capacity answers whether state may fit; bandwidth constrains low-intensity traffic; compute throughput constrains sufficiently high-intensity work; interconnect matters only when communication crosses devices. Marketing AI TOPS and a specific dense BF16 workload are not automatically comparable.


## 0. Predict before running

1. Classify each table field as capacity, bandwidth, compute, or connectivity.
2. Verify the 1792 GB/s arithmetic from width and pin rate.
3. Predict the Roofline ceiling at low versus high arithmetic intensity.

For each prediction, write the observation that would disprove it.


## 1. Theory and mechanism

The notebook audits a frozen official RTX 5090 fact set: 32 GB GDDR7, 512-bit interface, 1792 GB/s bandwidth, and compute capability 12.0. It verifies the width/rate bandwidth arithmetic, compares reported device capacity, imports the measured copy result from Lesson 07, and computes illustrative Roofline ceilings across arithmetic intensities. The supplied quick-table image is treated as an audit exercise; every value should be rechecked against a product page or architecture guide before use.

- A number without precision and sparsity convention is incomplete.
- Capacity, bandwidth, compute, and interconnect constrain different workload regimes.
- Official theoretical specifications and empirical application results must remain separate columns.


## 2. Trace the mechanism

### Mechanism map

```mermaid
flowchart LR
  A["workload shape + precision"] --> B["capacity check"]
  A --> C["arithmetic intensity"]
  C --> D["bandwidth roof"]
  C --> E["compute roof"]
  A --> F["software + interconnect support"]
  B --> G["measured candidate"]
  D --> G
  E --> G
  F --> G
```


## 3. Inspect the visual boundary

![GPU parameter quick table to audit](../assets/NVIDIA_GPU_parameter_quick_table.png)


These are conceptual teaching diagrams. They explain the named data path and are not die-accurate schematics of a particular commercial GPU.


## 4. Inspect the execution environment

The next cell asserts CUDA, records GPU/PyTorch/CUDA identity, fixes the seed, and defines the common event-timing helpers.


In [1]:
LESSON_NO = 17
LESSON_TITLE = 'Reading GPU Specification Tables Critically'

from pathlib import Path
from collections import Counter, deque
import json, math, platform, statistics, sys, time

import torch
import torch.nn.functional as F

assert torch.cuda.is_available(), "Chapter 04 retained runs require a CUDA-capable GPU."
DEVICE = torch.device("cuda")
SEED = 20260813 + LESSON_NO
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

major, minor = torch.cuda.get_device_capability(0)
props = torch.cuda.get_device_properties(0)
ENV = {
    "gpu": torch.cuda.get_device_name(0),
    "compute_capability": f"{major}.{minor}",
    "torch": torch.__version__,
    "cuda_runtime": str(torch.version.cuda),
    "python": sys.version.split()[0],
    "seed": SEED,
}
print(json.dumps(ENV, indent=2))

def percentile(values, q):
    ordered = sorted(float(v) for v in values)
    pos = (len(ordered) - 1) * q
    lo, hi = math.floor(pos), math.ceil(pos)
    if lo == hi:
        return ordered[lo]
    return ordered[lo] * (hi - pos) + ordered[hi] * (pos - lo)

def cuda_samples(fn, warmup=5, repeats=20):
    for _ in range(warmup):
        fn()
    torch.cuda.synchronize()
    samples = []
    for _ in range(repeats):
        start = torch.cuda.Event(enable_timing=True)
        stop = torch.cuda.Event(enable_timing=True)
        start.record()
        fn()
        stop.record()
        stop.synchronize()
        samples.append(float(start.elapsed_time(stop)))
    return samples

def summary(samples):
    return {
        "median_ms": statistics.median(samples),
        "p95_ms": percentile(samples, 0.95),
        "samples_ms": samples,
    }


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.13.0+cu130",
  "cuda_runtime": "13.0",
  "python": "3.12.3",
  "seed": 20260830
}


## 5. Freeze the experiment

| Role | Frozen value |
|---|---|
| Baseline | an unlabeled screenshot treated as authoritative |
| Candidate | source-tagged fields plus arithmetic and environment checks |
| Held constant | official fact snapshot, unit conventions, and recorded GPU |
| Measurements | capacity agreement, bandwidth formula error, achieved ratio, and Roofline ceilings |
| Evidence | `capacity-model` |

**Experiment:** Audit official fields, empirical device facts, and a measured copy result.


## 6. Inspect the code

A source dictionary carries units and URLs. Assertions catch bandwidth arithmetic or memory-technology drift. The Roofline table labels its compute roof as illustrative rather than substituting AI TOPS for a specific precision peak.

Do not run until the code matches the frozen table.


In [2]:
official = {
    "source": "https://www.nvidia.com/en-us/geforce/graphics-cards/50-series/rtx-5090/",
    "memory": {"value": 32.0, "unit": "GiB", "technology": "GDDR7"},
    "interface": {"value": 512.0, "unit": "bit"},
    "pin_rate": {"value": 28.0, "unit": "Gb/s per pin", "derived_from_bandwidth": True},
    "bandwidth": {"value": 1792.0, "unit": "GB/s"},
    "compute_capability": {"value": "12.0", "unit": "major.minor"},
}
calculated_bandwidth = official["interface"]["value"] * official["pin_rate"]["value"] / 8
bandwidth_error = abs(calculated_bandwidth - official["bandwidth"]["value"]) / official["bandwidth"]["value"]
device_gib = props.total_memory / 2**30

lesson07_path = chapter = Path.cwd().parent
lesson07_matches = sorted(chapter.glob("07-*/artifacts/rtx5090-result.json"))
lesson07 = json.loads(lesson07_matches[0].read_text(encoding="utf-8")) if lesson07_matches else None
lesson07_fraction = (lesson07 or {}).get("metrics", {}).get("achieved_fraction")

illustrative_compute_tflops = 100.0
intensities = (0.25, 1, 4, 16, 64, 256)
roofline = {
    str(ai): min(illustrative_compute_tflops, official["bandwidth"]["value"] * ai / 1000)
    for ai in intensities
}
fields_with_units = sum(
    1 for key, value in official.items()
    if isinstance(value, dict) and value.get("unit")
)
assert official["memory"]["technology"] == "GDDR7"
assert bandwidth_error < 1e-12
metrics = {
    "official_snapshot": official,
    "device_memory_gib": device_gib,
    "official_memory_gib": official["memory"]["value"],
    "capacity_difference_gib": device_gib - official["memory"]["value"],
    "calculated_bandwidth_gbps": calculated_bandwidth,
    "bandwidth_formula_error": bandwidth_error,
    "lesson07_achieved_fraction": lesson07_fraction,
    "fields_with_units": fields_with_units,
    "illustrative_compute_roof_tflops": illustrative_compute_tflops,
    "roofline_tflops_by_intensity": roofline,
}
fraction_text = "not available" if lesson07_fraction is None else f"{lesson07_fraction:.1%}"
analysis = (
    f"The width/rate formula reproduced {calculated_bandwidth:.0f} GB/s with zero arithmetic "
    f"error; the device reported {device_gib:.2f} GiB and Lesson 07 achieved {fraction_text} of "
    "the interface figure. The compute roof in the table is explicitly illustrative."
)
print(json.dumps(metrics, indent=2))


{
  "official_snapshot": {
    "source": "https://www.nvidia.com/en-us/geforce/graphics-cards/50-series/rtx-5090/",
    "memory": {
      "value": 32.0,
      "unit": "GiB",
      "technology": "GDDR7"
    },
    "interface": {
      "value": 512.0,
      "unit": "bit"
    },
    "pin_rate": {
      "value": 28.0,
      "unit": "Gb/s per pin",
      "derived_from_bandwidth": true
    },
    "bandwidth": {
      "value": 1792.0,
      "unit": "GB/s"
    },
    "compute_capability": {
      "value": "12.0",
      "unit": "major.minor"
    }
  },
  "device_memory_gib": 31.35833740234375,
  "official_memory_gib": 32.0,
  "capacity_difference_gib": -0.64166259765625,
  "calculated_bandwidth_gbps": 1792.0,
  "bandwidth_formula_error": 0.0,
  "lesson07_achieved_fraction": 0.8488019914992356,
  "fields_with_units": 5,
  "illustrative_compute_roof_tflops": 100.0,
  "roofline_tflops_by_intensity": {
    "0.25": 0.448,
    "1": 1.792,
    "4": 7.168,
    "16": 28.672,
    "64": 100.0,
    "256": 

## 7. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.13.0+cu130; CUDA runtime 13.0; Python 3.12.3.

| Measured field | Checked-in value |
|---|---:|
| Reported device memory | 31.3583 |
| Official capacity | 32.0000 |
| Bandwidth formula error | 0.00% |
| Lesson 07 achieved fraction | 84.88% |
| Fields with explicit units | 5 |


## 8. Explain rather than overclaim

The width/rate formula reproduced 1792 GB/s with zero arithmetic error; the device reported 31.36 GiB and Lesson 07 achieved 84.9% of the interface figure. The compute roof in the table is explicitly illustrative.

**Evidence boundary:** Measured environment facts feed explicit capacity or Roofline arithmetic. Declared hierarchy and resource fields remain assumptions until native counters confirm them.


## 9. Write the canonical artifact

The next cell stores the environment, metrics, analysis, evidence label, and bounded conclusion, then prints the exact JSON.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": 17, "title": 'Reading GPU Specification Tables Critically', "environment": ENV,
    "evidence_label": 'capacity-model', "metrics": metrics,
    "analysis": analysis, "conclusion": 'Choose hardware from a workload sheet that combines capacity, arithmetic intensity, latency/throughput targets, supported software, and measured evidence.',
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 17,
  "title": "Reading GPU Specification Tables Critically",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.13.0+cu130",
    "cuda_runtime": "13.0",
    "python": "3.12.3",
    "seed": 20260830
  },
  "evidence_label": "capacity-model",
  "metrics": {
    "official_snapshot": {
      "source": "https://www.nvidia.com/en-us/geforce/graphics-cards/50-series/rtx-5090/",
      "memory": {
        "value": 32.0,
        "unit": "GiB",
        "technology": "GDDR7"
      },
      "interface": {
        "value": 512.0,
        "unit": "bit"
      },
      "pin_rate": {
        "value": 28.0,
        "unit": "Gb/s per pin",
        "derived_from_bandwidth": true
      },
      "bandwidth": {
        "value": 1792.0,
        "unit": "GB/s"
      },
      "compute_capability": {
        "value": "12.0",
        "unit": "major.minor"
      }
    },
    "device_memory_gib": 31.35833740234375,
    "official_memory_gib": 32.0

## 10. Make the decision

> Choose hardware from a workload sheet that combines capacity, arithmetic intensity, latency/throughput targets, supported software, and measured evidence.

**Failure analysis:** Product pages can change, board variants differ, and theoretical peaks are not guaranteed. The checked-in snapshot must be revalidated for procurement or publication decisions.


## 11. Extend the evidence

Create a comparison sheet for two candidate GPUs using one frozen workload, then run the same memory and GEMM probes on both.

See [`README.md`](README.md) for the full explanation and references.
